# Inspectable ML readiness evidence
Read-only companion to the implemented checks in `src/safarsaathi_ml/readiness.py`. Run from the project/notebooks directory. No training, raw writes, uploads, or fabricated labels. See completion report for executed verification.


In [1]:
from pathlib import Path
import sys
project = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project / 'src'))
from safarsaathi_ml.readiness import data_readiness, load_arrivals
raw = project.parent / 'SafarSaathi_Datasets/raw'
report = data_readiness(project, raw, write=False)
{task: (item['readiness'], item['reason']) for task, item in report['models'].items()}


{'tourism_demand': ('READY_FOR_TRAINING',
  'Annual India arrivals support a small historical baseline; no current or destination-level coverage.'),
 'disaster_risk': ('NEEDS_LABELS',
  'UNDRR HIPs are hazard definitions, not observed events. No event dates, region outcomes, verified negative windows or impact labels.'),
 'flood_segmentation': ('NEEDS_LABELS',
  'Sentinel scenes and quality/cloud masks exist; no verified paired observed flood masks or event-separated labelled splits.'),
 'destination_stress': ('NEEDS_LABELS',
  'No observed stress, occupancy, crowd-density or aligned capacity time series. National arrivals and static OSM POIs do not establish destination stress.')}

In [2]:
observations, profile = load_arrivals(raw)
profile # grain, null rates, duplicates, coverage, source checksum and units


{'path': 'worldbank_un_tourism/API_ST.INT.ARVL_India_1995-2025.json',
 'format': 'World Bank JSON',
 'sha256': '6cd0c77c9560fcbf4c1514d89c53dcc0e9d077904d6156f38a87730ee95ac579',
 'rows': 31,
 'observed_rows': 25,
 'duplicate_keys': 0,
 'columns': {'destination': 'ISO3 string',
  'year': 'int',
  'arrivals': 'float/count'},
 'missing_rates': {'destination': 0.0,
  'year': 0.0,
  'arrivals': 0.1935483870967742},
 'missing_target_years': [2025, 2024, 2023, 2022, 2021, 2020],
 'observed_years': [1995, 2019],
 'geography': 'India national total; not Uttarakhand or individual destinations',
 'units': 'international tourist arrivals per calendar year; annual dates have no time zone',
 'imputation': 'none; trailing null targets excluded, internal gaps rejected',
 'source_updated': '2026-07-13'}

In [3]:
from safarsaathi_ml.forecasting import evaluate_forecast
evaluate_forecast(project) # saved artifact only, no fitting


{'model_id': 'tourism_demand_forecaster',
 'artifact_valid': True,
 'evaluation_reproduced': True,
 'metrics': {'by_destination': {'IND': {'MAE': 578589.4736842103,
    'RMSE': 710022.7072955804,
    'MASE': 2.1127419487686345,
    'WAPE': 0.036743304905390954,
    'n': 5}},
  'by_horizon': {'1': {'MAE': 578589.4736842103,
    'RMSE': 710022.7072955804,
    'MASE': 2.1127419487686345,
    'WAPE': 0.036743304905390954,
    'n': 5},
   '2': {'MAE': 1080394.7368421033,
    'RMSE': 1190166.1805798877,
    'MASE': 3.945103368750508,
    'WAPE': 0.06602870813397117,
    'n': 4},
   '3': {'MAE': 1572175.4385964882,
    'RMSE': 1719247.848706409,
    'MASE': 5.740859713184881,
    'WAPE': 0.09269902350215144,
    'n': 3}},
  'predictions': [{'destination': 'IND',
    'origin_year': 2014,
    'year': 2015,
    'horizon': 1,
    'actual': 13284000.0,
    'prediction': np.float64(13685052.631578948)},
   {'destination': 'IND',
    'origin_year': 2014,
    'year': 2016,
    'horizon': 2,
    'actu